In [5]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [20]:
from pyspark.sql import SparkSession

# Membuat SparkSession untuk Tugas Mandiri
spark = SparkSession.builder \
    .appName("Tugas7_2505060006_VandraNoprianusSembiring") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi log yang tidak perlu agar tampilan lebih rapi
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession untuk Tugas Mandiri siap!")
print("Versi Spark:", spark.version)

SparkSession untuk Tugas Mandiri siap!
Versi Spark: 3.5.9


In [22]:
# Membaca data dari HDFS
# A. EXTRACT - Membaca data dari HDFS
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True,
    inferSchema=True
)

df_cabang = spark.read.json(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json"
)

df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True,
    inferSchema=True
)

# Verifikasi
print("Transaksi:", df_transaksi.count(), "baris")
print("Cabang:", df_cabang.count(), "baris")
print("Menu:", df_menu.count(), "baris")

df_transaksi.printSchema()
df_cabang.printSchema()
df_menu.printSchema()

Transaksi: 4000 baris
Cabang: 3 baris
Menu: 15 baris
root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)

root
 |-- cabang_id: long (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)

root
 |-- menu_id: integer (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)



In [23]:
# B. TRANSFORM - Join & tambah kolom total_penjualan
df_gabung = df_transaksi \
    .join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner") \
    .withColumn("total_penjualan", col("qty") * col("harga"))

# Tampilkan 5 baris
df_gabung.show(5)

# Cek struktur
df_gabung.printSchema()

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [41]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, rank, row_number, col

# Membuat total penjualan
df_gabung = df_gabung.withColumn(
    "total_penjualan",
    col("qty") * col("harga")
)

# Ringkas total penjualan per cabang + menu
ringkasan_menu = df_gabung.groupBy(
    "nama_cabang", "nama_menu","menu_id"
).agg(
    sum("total_penjualan").alias("total_penjualan")
)

# Window berdasarkan masing-masing cabang
window_cabang = Window.partitionBy(
    "nama_cabang"
).orderBy(
    col("total_penjualan").desc()
)

# RANK dan ROW_NUMBER
hasil = ringkasan_menu \
    .withColumn("rank", rank().over(window_cabang)) \
    .withColumn("row_number", row_number().over(window_cabang))

hasil.show(5)

+--------------------+---------+-------+---------------+----+----------+
|         nama_cabang|nama_menu|menu_id|total_penjualan|rank|row_number|
+--------------------+---------+-------+---------------+----+----------+
|Kopi Nusantara Ma...|   Menu-1|      1|        8505000|   1|         1|
|Kopi Nusantara Ma...|  Menu-12|     12|        8085000|   2|         2|
|Kopi Nusantara Ma...|   Menu-3|      3|        7800000|   3|         3|
|Kopi Nusantara Ma...|   Menu-9|      9|        7665000|   4|         4|
|Kopi Nusantara Ma...|  Menu-15|     15|        7380000|   5|         5|
+--------------------+---------+-------+---------------+----+----------+
only showing top 5 rows



In [43]:
df_gabung.createOrReplaceTempView("gabungan")

total_barista = spark.sql("""
    SELECT
        kepala_barista,
        nama_cabang,
        SUM(total_penjualan) AS total_penjualan,
        COUNT(trx_id) AS jumlah_transaksi,
        AVG(total_penjualan) AS rata_rata_transaksi
    FROM gabungan
    GROUP BY kepala_barista, nama_cabang
    ORDER BY total_penjualan DESC
""")

total_barista.show()

+--------------+--------------------+---------------+----------------+-------------------+
|kepala_barista|         nama_cabang|total_penjualan|jumlah_transaksi|rata_rata_transaksi|
+--------------+--------------------+---------------+----------------+-------------------+
|         Nadia|Kopi Nusantara Se...|       90542000|            1393|   64997.8463747308|
|         Nadia|Kopi Nusantara Ma...|       87138000|            1302|  66926.26728110599|
|          Reza|Kopi Nusantara Yo...|       84645000|            1305|  64862.06896551724|
+--------------+--------------------+---------------+----------------+-------------------+



In [46]:
# E. LOAD - Simpan df_gabung ke HDFS sebagai Parquet,
# dipartisi per kategori menu

df_gabung.write.mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")

# Verifikasi
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan/

Found 4 items
-rw-r--r--   3 vandrasembiring supergroup          0 2026-10-01 00:12 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - vandrasembiring supergroup          0 2026-10-01 00:12 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - vandrasembiring supergroup          0 2026-10-01 00:12 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - vandrasembiring supergroup          0 2026-10-01 00:12 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


In [48]:
from pyspark.sql.functions import sum as spark_sum, avg, count, col

ringkasan = df_gabung.groupBy("nama_cabang").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_transaksi")
).orderBy(
    col("total_penjualan").desc()
)

ringkasan.show(truncate=False)

+-------------------------+---------------+----------------+-------------------+
|nama_cabang              |total_penjualan|jumlah_transaksi|rata_rata_transaksi|
+-------------------------+---------------+----------------+-------------------+
|Kopi Nusantara Semarang  |90542000       |1393            |64997.8463747308   |
|Kopi Nusantara Magelang  |87138000       |1302            |66926.26728110599  |
|Kopi Nusantara Yogyakarta|84645000       |1305            |64862.06896551724  |
+-------------------------+---------------+----------------+-------------------+



In [50]:
ringkasan_menu.filter(
    col("nama_cabang") == "Kopi Nusantara Yogyakarta"
).orderBy(
    col("total_penjualan").desc()
).show()

+--------------------+---------+-------+---------------+
|         nama_cabang|nama_menu|menu_id|total_penjualan|
+--------------------+---------+-------+---------------+
|Kopi Nusantara Yo...|  Menu-12|     12|        9275000|
|Kopi Nusantara Yo...|  Menu-15|     15|        8160000|
|Kopi Nusantara Yo...|   Menu-9|      9|        7175000|
|Kopi Nusantara Yo...|   Menu-1|      1|        7140000|
|Kopi Nusantara Yo...|   Menu-3|      3|        6390000|
|Kopi Nusantara Yo...|  Menu-13|     13|        6335000|
|Kopi Nusantara Yo...|   Menu-8|      8|        6240000|
|Kopi Nusantara Yo...|   Menu-5|      5|        5170000|
|Kopi Nusantara Yo...|  Menu-11|     11|        4488000|
|Kopi Nusantara Yo...|   Menu-6|      6|        4378000|
|Kopi Nusantara Yo...|   Menu-2|      2|        4356000|
|Kopi Nusantara Yo...|   Menu-4|      4|        4180000|
|Kopi Nusantara Yo...|  Menu-14|     14|        3906000|
|Kopi Nusantara Yo...|   Menu-7|      7|        3816000|
|Kopi Nusantara Yo...|  Menu-10

In [51]:
ringkasan.orderBy(
    col("total_penjualan").asc()
).show()

+--------------------+---------------+----------------+-------------------+
|         nama_cabang|total_penjualan|jumlah_transaksi|rata_rata_transaksi|
+--------------------+---------------+----------------+-------------------+
|Kopi Nusantara Yo...|       84645000|            1305|  64862.06896551724|
|Kopi Nusantara Ma...|       87138000|            1302|  66926.26728110599|
|Kopi Nusantara Se...|       90542000|            1393|   64997.8463747308|
+--------------------+---------------+----------------+-------------------+

